In [ ]:
from FIT_python.plot_style import apply_style
apply_style()


In [2]:
# %% 
import time
from pathlib import Path
import pandas as pd
import numpy as np
from tqdm.notebook import tqdm
from tqdm_joblib import tqdm_joblib
from sklearn.model_selection import ParameterSampler
from FIT_python.config import SPLITS_DIR ,RESULTS_DATA_DIR
from sklearn.model_selection import ParameterSampler
from FIT_python.data_split_and_summary.data_import_utils import load_and_prep_df_individual, get_feature_cols
from FIT_python.pipeline_individual_id.pairwise_individual_id_pipeline import (
    run_all_pairwise_projections_parallel
)
from FIT_python.pipeline_individual_id.generate_trails_and_trailpairs import (
    generate_pairwise_comparisons_from_df
)

import matplotlib.pyplot as plt
import seaborn as sns

# %% 1) Basis-Pfade & Hyperparameter-Räume (unverändert)
splits_root = Path("data/splits/eurasian_otter")
models_root = RESULTS_DATA_DIR / "eurasian_otter_random_search_standard_metrics" / "best_maj_test_pct"
output_root = Path("results/data/pairwise_comp_eurasian_otter")
output_root.mkdir(parents=True, exist_ok=True)

# %% 2) Statt Random-Sampling nur die Standard-Methoden
# Standard-Feature-Counts und feste Methode: Forward + LDA
std_feature_counts   = [10, 12, 14, 16, 18, 20]
std_selection_method = "forward"
std_reducer          = "lda"
std_n_components     = 2     # bei LDA nicht relevant
std_outlier_method   = None
std_scaler_method    = None

# Erzeuge das reine Standard-Grid
sampled_grid = (
    pd.DataFrame({"feature_count": std_feature_counts})
      .assign(
          selection_method=std_selection_method,
          reducer=std_reducer,
          n_components=std_n_components,
          outlier_method=std_outlier_method,
          scaler_method=std_scaler_method,
      )
      .merge(pd.DataFrame({"use_sex": [False, True]}), how="cross")
)
sampled_grid.sort_values("feature_count", ascending=False, inplace=True)
sampled_grid.reset_index(drop=True, inplace=True)

# Überprüfen
print(f"Standard grid shape: {sampled_grid.shape}")
display(sampled_grid)



# 6) Loop über Spezies
species_dir = Path(SPLITS_DIR)/"eurasian_otter"
species = "eurasian_otter"
print(f"\n🔄 Processing species: {species}")

# 6.1) Train/Test laden & Fold entfernen
df = load_and_prep_df_individual(species, Path(SPLITS_DIR)) #falls hier bugs valid vales f und m checken


# 3.2) Feature-Spalten bestimmen
feature_cols = get_feature_cols(df)
print(f"  → {len(feature_cols)} numerische Features")

# direkt vor generate_pairwise_comparisons_from_df
print("Unique individual_id:", df["individual_id"].nunique())
print("Unique trail (fallback_col):", df["trail"].nunique())

# 3.3) Pairs & Summary generieren
comps, summary = generate_pairwise_comparisons_from_df(df,num_individuals=5
)


print("  Summary der Trail-Längen:")
display(summary)

# 3.4) Resume-Logik für species-spezifisches Parquet
results_fp = output_root / f"{species}_all_results_combined.parquet"
if results_fp.exists():
    done = pd.read_parquet(results_fp, columns=sampled_grid.columns)
    done_keys = set(tuple(r) for r in done.values)
    todo = sampled_grid[
        sampled_grid.apply(lambda row: tuple(row.values) not in done_keys, axis=1)
    ].reset_index(drop=True)
else:
    todo = sampled_grid.copy()

print(f"  → Noch offen: {len(todo)}/{len(sampled_grid)} Runs")

# 3.5) Hyper-Parameter-Loop
for idx, row in todo.iterrows():
    # Spalten heißen jetzt "feature_count" (nicht feature_counts)
    n_features      = int(row.feature_count)
    selection       = row.selection_method
    reducer         = row.reducer
    n_components    = int(row.n_components)
    outlier_method  = row.outlier_method
    scaler_method   = row.scaler_method
    use_sex_flag    = bool(row.use_sex)

    desc = (
        f"FC={n_features}|FS={selection}|Red={reducer}/{n_components}"
        f"|Out={outlier_method}|Scl={scaler_method}|Sex={'on' if use_sex_flag else 'off'}"
    )
    print(f"    → Run {idx+1}/{len(todo)}: {desc}")
    t0 = time.time()

    # Outlier- und Scaler-Parameter in Listen verpacken
    outlier_list = [outlier_method] if outlier_method is not None else [None]
    scaler_list  = [scaler_method]  if scaler_method  is not None else [None]

    with tqdm_joblib(tqdm(desc=desc, total=len(comps), leave=False)):
        results = run_all_pairwise_projections_parallel(
            comparisons=comps,
            df=df,
            feature_cols=feature_cols,
            k_features=n_features,
            selection_method=selection,
            reducers=[reducer],
            n_components=[n_components],
            outlier_methods=outlier_list,
            scaler_methods=scaler_list,
            use_sexmodel_prediction=use_sex_flag,
            sexmodel_path=(models_root / f"{species}.joblib") if use_sex_flag else None,
            debug=False,
            n_jobs=-1
        )
    print(f"      → {len(results)} Rows in {time.time()-t0:.1f}s")

    # 3.6) Results anreichern & speichern
    df_run = pd.DataFrame(results)
    for col in sampled_grid.columns:
        df_run[col] = row[col]
    if results_fp.exists():
        df_full = pd.concat([pd.read_parquet(results_fp), df_run], ignore_index=True)
    else:
        df_full = df_run
    df_full.to_parquet(results_fp, index=False)

print(f"  ✅ Fertig für {species}, Parquet: {results_fp}")

# %% 4) Ergebnisse
all_parquets = [results_fp]
df_all = pd.concat((pd.read_parquet(p) for p in all_parquets), ignore_index=True)
df_all.to_parquet(output_root / 'eurasian_otter_all_results_combined.parquet', index=False)
print("\n🎉 Fertig – Ergebnis-Parquet erzeugt:")
print(f"   {output_root/'eurasian_otter_all_results_combined.parquet'}")
display(df_all.head())


Standard grid shape: (12, 7)


,feature_count,selection_method,reducer,n_components,outlier_method,scaler_method,use_sex
0,20,forward,lda,2,None,None,True
1,20,forward,lda,2,None,None,False
2,18,forward,lda,2,None,None,True
3,18,forward,lda,2,None,None,False
4,16,forward,lda,2,None,None,False
5,16,forward,lda,2,None,None,True
6,14,forward,lda,2,None,None,False
7,14,forward,lda,2,None,None,True
8,12,forward,lda,2,None,None,False
9,12,forward,lda,2,None,None,True



🔄 Processing species: eurasian_otter
  → 205 numerische Features
Unique individual_id: 27
Unique trail (fallback_col): 54
  Summary der Trail-Längen:


,sub_size,n_animals,n_trails,avg_comp_per_ind_same,sd_comp_per_ind_same,avg_comp_per_ind_diff,sd_comp_per_ind_diff,ratio_same_to_diff
0,10,5,25,NaN,NaN,NaN,NaN,NaN
1,Total,5,300,10.0,0.0,100.0,0.0,50.0


  → Noch offen: 12/12 Runs
    → Run 1/12: FC=20|FS=forward|Red=lda/2|Out=None|Scl=None|Sex=on


...

  0%|          | 0/300 [00:00<?, ?it/s]

  0%|          | 0/300 [00:00<?, ?it/s]

      → 300 Rows in 28.7s


Processing Pairs:   0%|          | 0/300 [00:28<?, ?it/s]


ArrowTypeError: ("Expected bytes, got a 'bool' object", 'Conversion failed for column same_individual with type object')

In [ ]:
df_all

In [ ]:
import pandas as pd
import seaborn as sns  # nur für die Heatmap-Matrix

df_pairs = pd.DataFrame(comparisons)
mat = pd.crosstab(df_pairs.ind_a, df_pairs.ind_b)
sns.heatmap(mat, annot=True, fmt="d")
plt.title("Pair-Count Heatmap")
plt.show()